# TD3 evaluation

Pre-fix outputs are preserved in `../archive/notebooks/` under this section name. These updated cells have no historical execution output. Run with the project Python environment. Training is opt-in; old models are never loaded automatically.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rl_project").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import Video, display
from Continuous_Diff_Drive.experiments import latest_checkpoint, load_for_evaluation, ENVIRONMENT, FIXED_OBSTACLES


In [ ]:
ALGORITHM = "td3"
OUTPUT_ROOT = ROOT / "artifacts"
CHECKPOINT_PATH = latest_checkpoint(ALGORITHM, OUTPUT_ROOT)  # Only completed new runs.
ALLOW_LEGACY_NORMALIZED = False  # DDPG evaluation only; requires verified historical scaling.
RECORD = False
EVALUATION_SEED = 1234
RANDOM_EPISODES = 10
ENV_KWARGS = {**ENVIRONMENT, "random_obst": False, "obstacles": FIXED_OBSTACLES}


## Fixed and seeded random layouts
The fixed deterministic scenario is evaluated once; repeated identical rollouts are not independent evidence. Random layouts are sampled from the training mixture with different seeds, not claimed to be outside that distribution. Missing or incompatible checkpoints are reported explicitly.

In [ ]:
results = []
for random_layout, count in ((False, 1), (True, RANDOM_EPISODES)):
    config = {**ENV_KWARGS, "random_obst": random_layout}
    agent, status = load_for_evaluation(ALGORITHM, CHECKPOINT_PATH, config,
                                       allow_legacy_normalized=ALLOW_LEGACY_NORMALIZED)
    print("random" if random_layout else "fixed", status)
    if agent is not None:
        try:
            result = agent.evaluate(episodes=count, seed=EVALUATION_SEED,
                                     output_root=OUTPUT_ROOT, record=RECORD)
            results.append(result)
            print(result.episodes)
            print(result.artifacts)
        finally:
            agent.env.close()


In [ ]:
for result in results:
    for path in sorted((result.run_dir / "videos/evaluation").glob("*.mp4")):
        display(Video(filename=str(path), embed=True))
